# MoS2 RPA analysis and presentation plots

This notebook collects the material's electronic-structure, momentum-resolved RPA polarization, equilibrium method comparison, and effective dielectric-response plots.

In [ ]:
from pathlib import Path
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "figure.figsize": (7.1, 4.35),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "pdf.fonttype": 42,
})

# Legacy settings retained below are overridden by the thesis style above.
plt.rcParams.update({
    "figure.figsize": (8.8, 5.0), "figure.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelsize": 12, "axes.titlesize": 14,
    "legend.frameon": False, "legend.fontsize": 9, "savefig.dpi": 220,
})

def find_w90_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for parent in (start, *start.parents):
        if (parent / "data_analysis").is_dir() and (parent / "mos2").is_dir():
            return parent
    raise RuntimeError("Could not find w90 root.")

def load(path, *, mmap=False):
    return np.load(path, mmap_mode="r" if mmap else None, allow_pickle=True)

def nearest_index(values, target):
    return int(np.argmin(np.abs(values - target)))

def export_figure(fig, filename):
    path = PRESENTATION_OUTPUTS / filename
    fig.savefig(path, bbox_inches="tight")
    print("Exported:", path)
    return path

W90_ROOT = find_w90_root()
DATA_ANALYSIS_ROOT = W90_ROOT / "data_analysis"
VALIDATION_ROOT = DATA_ANALYSIS_ROOT / "validation_outputs/mos2"
PRESENTATION_OUTPUTS = DATA_ANALYSIS_ROOT / "presentation_outputs" / "mos2"
PRESENTATION_OUTPUTS.mkdir(parents=True, exist_ok=True)
print("Presentation outputs:", PRESENTATION_OUTPUTS)

## 1. Electronic-structure reference

In [ ]:
# Direct diagonalization of the supplied 22-orbital rectangular-cell Hamiltonian.
# All 22 bands are retained; the plot is only zoomed in energy around midgap.
BAND_DATA = VALIDATION_ROOT / "band_structure" / "validation_mos2_rectangular_cell_GXSYG_midgap.npz"
band = load(BAND_DATA)
k_path = band["k_points"]
band_energies = band["plotted_eigenvalues"]
ticks = band["tick_positions"]
labels = band["tick_labels"]
fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
for values in band_energies.T:
    ax.plot(k_path, values, color="#1f4e79", linewidth=0.9)
ax.axhline(0.0, color="#b22222", linestyle="--", linewidth=1.1)
ax.set_xticks(ticks)
ax.set_xticklabels([r"$\Gamma$" if label == "G" else label for label in labels])
for tick in ticks: ax.axvline(tick, color="#666666", alpha=0.25, linewidth=0.7)
ax.set(xlabel="Wave-vector path", ylabel=r"$E-E_\mathrm{midgap}$ (eV)", title="MoS$_2$ Orthogonal-Cell Band Structure")
ax.set_xlim(float(k_path[0]), float(k_path[-1])); ax.set_ylim(-3, 3)
ax.grid(True, alpha=0.25, linewidth=0.5)
export_figure(fig, "mos2_rectangular_cell_bandstructure_GXSYG_midgap.png")
plt.show()

## 2. Momentum-resolved equilibrium RPA polarization

Each figure shows the real and imaginary parts of the periodic RPA polarization
at one finite momentum transfer. These are direct q-resolved responses, not
q-averaged traces.

In [ ]:
POLARIZATION_DATA = VALIDATION_ROOT / "polarization_behavior" / "validation_mos2_rpa_polarization.npz"
pol = load(POLARIZATION_DATA)
q_points = pol["q_points"]
frequencies_eV = pol["frequencies"]
p_base = pol["p_base"]

for fraction, q_text, filename in (
    (0.25, r"$q=\pi/4$", "mos2_rpa_polarization_q_pi_over_4.png"),
    (0.50, r"$q=\pi/2$", "mos2_rpa_polarization_q_pi_over_2.png"),
):
    q_index = nearest_index(q_points, fraction * np.pi)
    fig, ax = plt.subplots(figsize=(7.1, 4.25), constrained_layout=True)
    ax.plot(
        frequencies_eV,
        p_base[q_index].real,
        color="#1f5a91",
        linewidth=1.8,
        label=r"$\operatorname{Re}P(q,\omega)$",
    )
    ax.plot(
        frequencies_eV,
        p_base[q_index].imag,
        color="#d95f02",
        linewidth=1.8,
        label=r"$\operatorname{Im}P(q,\omega)$",
    )
    ax.axhline(0.0, color="0.35", linewidth=0.75)
    ax.set_title(rf"MoS$_2$ RPA Polarization at {q_text}")
    ax.set_xlabel(r"Energy transfer, $\hbar\omega$ (eV)")
    ax.set_ylabel(r"Polarization response, $P(q,\omega)$ (arb. units)")
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.legend()
    export_figure(fig, filename)
    plt.show()


## 3. Equilibrium polarization: finite-device GW/NEGF versus periodic RPA

This is an optional legacy diagnostic. It is shown only when a complete matched
MoS2 RPA and finite-device GW/NEGF output pair is available. Missing files do
not affect the momentum-resolved RPA polarization or dielectric results in the
following sections.

In [ ]:
PAIR_CANDIDATES = [
    (
        "stable_inputs clean equilibrium pair",
        DATA_ANALYSIS_ROOT / "generated_equilibrium_comparisons/mos2/outputs_RPA_export_stable_inputs/raw_rpa_debug",
        DATA_ANALYSIS_ROOT / "generated_equilibrium_comparisons/mos2/outputs_NEGF_equilibrium_stable_inputs/p_retarded_diagonal_0.npy",
    ),
    (
        "older existing pair",
        DATA_ANALYSIS_ROOT / "generated_equilibrium_comparisons/mos2/outputs_RPA_export/raw_rpa_debug",
        DATA_ANALYSIS_ROOT / "generated_equilibrium_comparisons/mos2/outputs_NEGF_equilibrium/p_retarded_diagonal_0.npy",
    ),
]

selected = next(
    (
        (label, rpa_dir, negf_file)
        for label, rpa_dir, negf_file in PAIR_CANDIDATES
        if (rpa_dir / "polarization_retarded_qw.npy").exists()
        and negf_file.exists()
    ),
    None,
)

if selected is None:
    print("Skipping optional MoS2 finite-device GW/NEGF comparison.")
    print("No complete matched RPA + GW/NEGF output pair was found:")
    for candidate_label, rpa_dir, negf_file in PAIR_CANDIDATES:
        print(f"\n{candidate_label}")
        print("  RPA:", rpa_dir / "polarization_retarded_qw.npy")
        print("  GW/NEGF:", negf_file)
else:
    label, RPA_RAW, NEGF_DIAGONAL = selected
    print("Using:", label)

    p_rpa = load(RPA_RAW / "polarization_retarded_qw.npy", mmap=True)
    freq = np.asarray(load(RPA_RAW / "frequencies_eV.npy"))
    p_negf = load(NEGF_DIAGONAL, mmap=True)

    rpa = np.asarray(np.trace(p_rpa, axis1=2, axis2=3)).mean(axis=0)
    cell_ratio = p_negf.shape[1] / p_rpa.shape[-1]
    negf = np.asarray(p_negf.sum(axis=1)) / cell_ratio

    n = min(len(freq), len(rpa), len(negf))
    x = freq[:n]
    rpa = rpa[:n]
    negf = negf[:n]
    print("Finite-device/unit-cell basis ratio:", cell_ratio)

    fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
    ax.plot(
        x, negf.real, color="black", linewidth=1.9,
        label=f"Re GW/NEGF / {cell_ratio:g} cells",
    )
    ax.plot(
        x, negf.imag, color="black", linestyle="--", linewidth=1.9,
        label=f"Im GW/NEGF / {cell_ratio:g} cells",
    )
    ax.plot(x, rpa.real, color="tab:blue", linewidth=1.9, label="Re q-avg RPA")
    ax.plot(
        x, rpa.imag, color="tab:blue", linestyle="--", linewidth=1.9,
        label="Im q-avg RPA",
    )
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set(
        title=r"MoS$_2$ Equilibrium Polarization: GW/NEGF and RPA",
        xlabel=r"Energy transfer, $\hbar\omega$ (eV)",
        ylabel="Polarization trace per unit cell",
    )
    ax.legend(ncol=2)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    export_figure(fig, "mos2_equilibrium_polarization_negf_vs_rpa.png")
    plt.show()


## 4. Absolute-magnitude validation: periodic RPA versus periodic Green-function bubble

This controlled comparison evaluates the polarization using two independent
formulations with the same periodic MoS2 Bloch Hamiltonian,
occupations, q-points, broadening, density vertex, and spin degeneracy:

- the implemented band-sum RPA;
- the equilibrium Green-function bubble underlying the GW polarization.

No normalization or fitted scaling is applied to either curve. Agreement therefore
validates the RPA complex response and its absolute magnitude.


In [ ]:
PERIODIC_VALIDATION = PRESENTATION_OUTPUTS / "mos2_periodic_rpa_vs_gf_bubble.npz"
if not PERIODIC_VALIDATION.exists():
    raise FileNotFoundError(
        "Missing periodic RPA/GF validation data. Run "
        "data_analysis/scripts/validate_periodic_rpa_vs_gf_bubble.py mos2 first."
    )

periodic_validation = load(PERIODIC_VALIDATION)
validation_frequency = periodic_validation["frequencies_eV"]
validation_q_over_pi = periodic_validation["q_over_pi"]
validation_rpa = periodic_validation["rpa_trace"]
validation_gf = periodic_validation["gf_trace"]

fig, axes = plt.subplots(
    len(validation_q_over_pi), 2, figsize=(12.8, 8.2), sharex=True,
    constrained_layout=True,
)

for row, q_fraction in enumerate(validation_q_over_pi):
    direct_error = np.linalg.norm(
        validation_gf[row] - validation_rpa[row]
    ) / np.linalg.norm(validation_rpa[row])
    direct_scale = np.vdot(
        validation_gf[row], validation_rpa[row]
    ).real / np.vdot(validation_gf[row], validation_gf[row]).real

    for column, component in enumerate(("real", "imag")):
        axis = axes[row, column]
        rpa_component = getattr(validation_rpa[row], component)
        gf_component = getattr(validation_gf[row], component)
        component_error = np.linalg.norm(
            gf_component - rpa_component
        ) / np.linalg.norm(rpa_component)
        correlation = np.corrcoef(gf_component, rpa_component)[0, 1]
        axis.plot(
            validation_frequency, rpa_component, linewidth=2.1,
            label="Band-sum RPA",
        )
        axis.plot(
            validation_frequency, gf_component, "--", linewidth=1.8,
            label="Periodic GF bubble",
        )
        axis.axhline(0.0, color="0.35", linewidth=0.7)
        axis.set_title(
            f"{component.capitalize()} response, q/pi={q_fraction:.2f}\n"
            f"direct error = {100 * component_error:.3f}%"
        )
        axis.set_ylabel("Polarization trace")
        axis.grid(True, alpha=0.25, linewidth=0.5)
        print(
            f"q/pi={q_fraction:.2f} {component}: "
            f"relative error={component_error:.6e}, correlation={correlation:.8f}"
        )

    print(
        f"q/pi={q_fraction:.2f} full complex response: "
        f"direct error={direct_error:.6e}, diagnostic scale={direct_scale:.8f}"
    )

for axis in axes[-1]:
    axis.set_xlabel("Energy transfer (eV)")
axes[0, 0].legend()
fig.suptitle("MoS2 Absolute-Magnitude Validation: Periodic RPA vs GF Bubble")
export_figure(fig, "mos2_periodic_rpa_vs_gf_bubble.png")
plt.show()


## 5. Effective RPA dielectric response

The effective scalar dielectric response uses the saved dominant-positive
Coulomb-mode projection. Direct reconstruction of the projected Coulomb
interaction from the MoS2 input matrix confirms that this dataset already uses
the unscaled interaction, corresponding to \(\epsilon_r=1\).

In [ ]:
DIELECTRIC_DATA = VALIDATION_ROOT / "dielectric_function" / "validation_mos2_rpa_effective_dielectric.npz"
diel = load(DIELECTRIC_DATA)
q_points_d = diel["q_points"]
frequencies_d = diel["frequencies"]
epsilon_eff_epsilon1 = diel["epsilon_eff"]
projection = str(diel["projection"]) if "projection" in diel.files else "effective projection"

for fraction, q_text, filename in (
    (0.25, r"$q=\pi/4$", "mos2_dielectric_epsilon1_q_pi_over_4.png"),
    (0.50, r"$q=\pi/2$", "mos2_dielectric_epsilon1_q_pi_over_2.png"),
):
    q_index = nearest_index(q_points_d, fraction * np.pi)
    fig, ax = plt.subplots(figsize=(7.1, 4.25), constrained_layout=True)
    ax.plot(
        frequencies_d,
        epsilon_eff_epsilon1[q_index].real,
        color="#1f5a91",
        linewidth=1.8,
        label=r"$\operatorname{Re}\epsilon_{\mathrm{eff}}(q,\omega)$",
    )
    ax.plot(
        frequencies_d,
        epsilon_eff_epsilon1[q_index].imag,
        color="#d95f02",
        linewidth=1.8,
        label=r"$\operatorname{Im}\epsilon_{\mathrm{eff}}(q,\omega)$",
    )
    ax.axhline(1.0, color="0.45", linewidth=0.8, linestyle=":")
    ax.axhline(0.0, color="0.35", linewidth=0.75)
    ax.set_title(
        rf"MoS$_2$ Effective Dielectric Response at {q_text} "
        rf"($\epsilon_r=1$)"
    )
    ax.set_xlabel(r"Energy transfer, $\hbar\omega$ (eV)")
    ax.set_ylabel(r"Effective dielectric response, $\epsilon_{\mathrm{eff}}$")
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.legend()
    export_figure(fig, filename)
    plt.show()

print("Projection:", projection)


## Presentation summary

The q-resolved polarization and dielectric figures are direct outputs of the
completed periodic RPA calculation for the supplied 22-orbital orthogonal-cell
MoS2 Hamiltonian. They can demonstrate the implementation's response for that
input model, but they should not be presented as a literature-quality primitive-
cell MoS2 validation because the supplied Hamiltonian and its band-path
interpretation remain an important modeling caveat.

## 6. Static 2D polarizability and literature comparison

For an isolated two-dimensional semiconductor, a unique bulk dielectric
constant is not defined without choosing a material thickness. The
thickness-independent quantity is the static 2D electronic polarizability,

\[
\alpha_{2D}(q)=
-\frac{e^2 P_{\mathrm{total}}(q,0)}
       {A_{\mathrm{cell}}q^2},
\qquad
\alpha_{2D}=\lim_{q\rightarrow0}\alpha_{2D}(q).
\]

The calculation uses the full two-dimensional orthogonal-cell Hamiltonian,
\(A_{\mathrm{cell}}=17.5152\ \mathrm{\AA}^2\), a \(320\times160\) k-grid,
14 occupied bands, and five small-q points in the extrapolation. It includes
the electronic head response only; local-field, ionic, and spin-orbit effects
are not included.

Two useful literature references are:

- Berkelbach, Hybertsen, and Reichman, *Phys. Rev. B* **88**, 045318
  (2013): \(\alpha_{2D}=6.60\ \mathrm{\AA}\).
- Rasmussen and Thygesen, *J. Phys. Chem. C* **119**, 13169 (2015):
  \(d\epsilon_{2D}/dq|_{q=0}=44.3\ \mathrm{\AA}\), corresponding to
  \(\alpha_{2D}=44.3/(2\pi)=7.05\ \mathrm{\AA}\).

The present value is larger because this Wannier model has a smaller
\(1.69\) eV gap and the diagnostic omits local-field and spin-orbit effects.
It should therefore be presented as a model-level comparison, not a
high-accuracy material prediction.

In [ ]:
STATIC_DATA = (
    DATA_ANALYSIS_ROOT
    / "validation_outputs/mos2/dielectric_function/static_2d"
    / "mos2_2d_static_final.npz"
)
static = load(STATIC_DATA)
q_static = static["q_magnitudes_inverse_angstrom"]
alpha_static = static["alpha_2d_by_q_angstrom"]
alpha_zero = float(static["alpha_2d_angstrom"])
slope = float(static["alpha_q2_slope_angstrom3"])
fit_count = int(static["fit_count"])
effective_thickness = float(static["effective_thickness_angstrom"])
effective_epsilon = float(static["effective_epsilon_electronic"])

fig, ax = plt.subplots(figsize=(7.1, 4.25), constrained_layout=True)
ax.plot(
    q_static, alpha_static, "o-", color="#1f5a91",
    label=r"$\alpha_{2D}(q)$",
)
q_fit = np.linspace(0.0, q_static[fit_count - 1], 150)
ax.plot(
    q_fit, alpha_zero + slope * q_fit**2, "--", color="#d95f02",
    label=rf"$q\to0$: $\alpha_{{2D}}={alpha_zero:.3f}$ Å",
)
ax.scatter([0.0], [alpha_zero], marker="x", color="black", s=55, zorder=3)
ax.set_title(r"MoS$_2$ Static Electronic 2D Polarizability")
ax.set_xlabel(r"Momentum transfer, $|q|$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(r"2D polarizability, $\alpha_{2D}$ ($\mathrm{\AA}$)")
ax.grid(True, alpha=0.25, linewidth=0.5)
ax.legend()
export_figure(fig, "mos2_static_electronic_2d_polarizability.png")
plt.show()

alpha_berkelbach = 6.60
alpha_rasmussen = 44.3 / (2.0 * np.pi)
print(f"Present alpha_2D: {alpha_zero:.3f} Angstrom")
print(f"Berkelbach et al.: {alpha_berkelbach:.3f} Angstrom")
print(f"Rasmussen & Thygesen: {alpha_rasmussen:.3f} Angstrom")
print(f"Difference from 6.60 A: {(alpha_zero/alpha_berkelbach-1)*100:.1f}%")
print(f"Difference from 7.05 A: {(alpha_zero/alpha_rasmussen-1)*100:.1f}%")
print(
    f"Optional epsilon_eff(d={effective_thickness:.2f} A): "
    f"{effective_epsilon:.2f}"
)
